In [1]:
# IMPORTSSSS
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Concatenate, RepeatVector, TimeDistributed

In [2]:

# Preprocess blocks
def analyze_stroke_data(csv_filepath):
    # 1. Load the data
    df = pd.read_csv(csv_filepath)
    
    # 2. Calculate time differences between rows (Delta Time / dt)
    df['dt'] = df['time'].diff().fillna(0)
    
    # 3. Calculate distance between points (Delta Distance using Pythagorean theorem)
    df['dx'] = df['x'].diff().fillna(0)
    df['dy'] = df['y'].diff().fillna(0)
    
    # 4. Calculate Velocity (Distance / Time)
    # np.where prevents division-by-zero errors if two events fire at the exact same millisecond
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2) 
    df['velocity'] = np.where(df['dt'] > 0, df['distance'] / df['dt'], 0)
    
    # Calculate "Writing Duration" 
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 1
    writing_duration = df['dt'].where(df['touching'] == 1, 0).sum()

    # Calculate "In-Air Pen Duration" (The pause time biomarker)
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 0
    in_air_duration = df['dt'].where(df['touching'] == 0, 0).sum()
    
    # TODO 3: Print the results!
    print(f"--- Analysis for: {csv_filepath} ---")
    print(f"Total Writing Duration : {writing_duration} ms")
    print(f"Total In-Air Pauses    : {in_air_duration} ms")
    print(f"Average Pen Velocity   : {df['velocity'].mean():.2f} px/ms")
    print("-" * 40)
    
    return df


In [3]:

def load_and_pad_data(data_dir):
    sequences = []
    latencies = []
    labels = []
    
    for filename in os.listdir(data_dir):
        if not filename.endswith('.csv'):
            continue
            
        # Grab the label (1 for dyslexia, 0 for normal)
        label = 1 if filename.startswith("dyslexia") else 0
        smart_labels = np.zeros((MAX_TIMESTEPS, 1))
        
        filepath = os.path.join(data_dir, filename)
       
        # loads the CSV and calculates the velocity/distances.
        df = analyze_stroke_data(filepath)
        
        actual_length = min(len(df), MAX_TIMESTEPS)
        
        if label == 1: # ONLY apply this logic if the file is from a Dyslexic sample!
            
            # Figure out what "slow" means for this specific patient
            # grab the bottom 25% of their writing speed to define a "stutter"
            moving_data = df[df['velocity'] > 0]['velocity']
            stutter_threshold = np.percentile(moving_data, 25) if len(moving_data) > 0 else 0.5
            
            for i in range(actual_length):
                # Anomaly A: The pen is lifted in the air (Hesitation)
                is_paused = (df['touching'].iloc[i] == 0)
                
                # Anomaly B: The pen is touching, but dragging very slowly (Micro-stutter)
                is_stuttering = (df['touching'].iloc[i] == 1) and (df['velocity'].iloc[i] < stutter_threshold)
                
                # If they are struggling at this exact millisecond, flag it!
                if is_paused or is_stuttering:
                    smart_labels[i] = 1 
                    
        
        if 'latency' in df.columns:
            latency_val = df['latency'].iloc[0]
        else:
            latency_val = 0
            
        # Extract just the features we want
        stroke_data = df[['velocity', 'pressure', 'touching']].values
        
        # Pad or Truncate to MAX_TIMESTEPS (500)
        if len(stroke_data) > MAX_TIMESTEPS:
            stroke_data = stroke_data[:MAX_TIMESTEPS] # Truncate if too long
        else:
            # Pad with zeros if too short
            padding = np.zeros((MAX_TIMESTEPS - len(stroke_data), FEATURES))
            stroke_data = np.vstack((stroke_data, padding))
            
        sequences.append(stroke_data)
        latencies.append(latency_val)
        # labels.append(label)
        # labels.append(np.full((MAX_TIMESTEPS, 1), label))
        labels.append(smart_labels)
        
    return np.array(sequences), np.array(latencies), np.array(labels)


In [4]:
# Hyperparameters
MAX_TIMESTEPS = 500  #  standardize all writing samples to 500 time-steps
FEATURES = 3         #  feed it with 3 features: [velocity, pressure, touching]

def build_model():
    sequence_input = Input(shape=(MAX_TIMESTEPS, FEATURES), name="kinematics")
    # padding='same' ensures the sequence stays exactly 500 steps long
    x = Conv1D(32, kernel_size=15, activation='relu', padding='same')(sequence_input)
    x = Conv1D(64, kernel_size=10, dilation_rate=2, activation='relu', padding='same')(x)
    # Latency INput
    latency_input = Input(shape=(1,), name="latency")
    # Stretch the 1 single latency number into 500 copies
    repeated_latency = RepeatVector(MAX_TIMESTEPS)(latency_input) 
    
    merged = Concatenate()([x, repeated_latency]) 
    
    y = TimeDistributed(Dense(64, activation='relu'))(merged)
    y = Dropout(0.5)(y)
    final_output = TimeDistributed(Dense(1, activation='sigmoid'))(y)
    
    model = Model(inputs=[sequence_input, latency_input], outputs=final_output)
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
    )
    return model

In [5]:
model = build_model()
model.summary()
print("Loading data...")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ kinematics          │ (None, 500, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 32)   │      1,472 │ kinematics[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latency             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 500, 64)   │     20,544 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 500, 1)    │          0 │ latency[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 500, 65)   │          0 │ conv1d_1[0][0],   │
│ (Concatenate)       │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 500, 64)   │      4,224 │ concatenate[0][0] │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 500, 64)   │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 500, 1)    │         65 │ dropout[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,305 (102.75 KB)

 Trainable params: 26,305 (102.75 KB)

 Non-trainable params: 0 (0.00 B)

Loading data...


In [6]:
# Point this to your data collector folder
X_seq, X_lat, y = load_and_pad_data("datasets/") 
print(f"Data loaded! \nShape of X_seq (timeseries sequence) : {X_seq.shape} \nShape of X_lat (latency) :{X_lat.shape} \nShape of y (labels) : {y.shape}")
    

--- Analysis for: datasets/dyslexia_A_1787797509.csv ---
Total Writing Duration : 2472.6999999955297 ms
Total In-Air Pauses    : 24.80000000447035 ms
Average Pen Velocity   : 0.67 px/ms
----------------------------------------
--- Analysis for: datasets/dyslexia_C_1787818486.csv ---
Total Writing Duration : 1714.2999999970198 ms
Total In-Air Pauses    : 8.299999997019768 ms
Average Pen Velocity   : 0.35 px/ms
----------------------------------------
--- Analysis for: datasets/dyslexia_A_1787797536.csv ---
Total Writing Duration : 9554.400000002235 ms
Total In-Air Pauses    : 67.29999999701977 ms
Average Pen Velocity   : 0.26 px/ms
----------------------------------------
--- Analysis for: datasets/dyslexia_A_1787818622.csv ---
Total Writing Duration : 2986.89999999851 ms
Total In-Air Pauses    : 25.799999997019768 ms
Average Pen Velocity   : 0.28 px/ms
----------------------------------------
--- Analysis for: datasets/normal_B_1787808264.csv ---
Total Writing Duration : 1139.300000000

In [7]:

# Count how many total 0s and 1s exist in the entire training set 'y'
total_zeros = np.sum(y == 0)
total_ones = np.sum(y == 1)
total_samples = total_zeros + total_ones

# Apply the 'Weight = Total_Samples / (Number_of_Classes * Samples_in_Class)' formula
weight_for_0 = total_samples / (2.0 * total_zeros)
weight_for_1 = total_samples / (2.0 * total_ones)

# 3. Create the exact dictionary
sample_weight = np.ones(shape=y.shape) * weight_for_0
sample_weight[y == 1] = weight_for_1

print(f"Calculated Weights -> 0 (normal): {weight_for_0:.2f}, 1 (dyslexic): {weight_for_1:.2f}")

Calculated Weights -> 0 (normal): 0.53, 1 (dyslexic): 10.42


In [8]:

print("Starting training...")
# history = model.fit(x_train, y_train, epochs=20, validation_split=0.2)
history = model.fit([X_seq, X_lat], y, epochs=20, validation_split=0.2, sample_weight=sample_weight)
print("Saving the model...")
model.save("models/elkinematicV2.keras")

Starting training...
Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step - accuracy: 0.6667 - loss: 7.8325 - recall: 0.3174 - val_accuracy: 0.9527 - val_loss: 7.9441 - val_recall: 0.0000e+00
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 615ms/step - accuracy: 0.6527 - loss: 7.7779 - recall: 0.3433 - val_accuracy: 0.9527 - val_loss: 7.9441 - val_recall: 0.0000e+00
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 699ms/step - accuracy: 0.6414 - loss: 7.8201 - recall: 0.3527 - val_accuracy: 0.9527 - val_loss: 7.9441 - val_recall: 0.0000e+00
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 623ms/step - accuracy: 0.6407 - loss: 7.9344 - recall: 0.3370 - val_accuracy: 0.9527 - val_loss: 7.9441 - val_recall: 0.0000e+00
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 556ms/step - accuracy: 0.6475 - loss: 7.8428 - recall: 0.3378 - val_accuracy: 0.9527 - val_loss: 7.9441 - val_recall: 0.0000e+00
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 535ms/step - accuracy: 0.6476 - loss: 7.8655 - recall: 0.3339 - val_accuracy: 0.9527 - val_loss: 7.944